In [ ]:
# Example 1: Construct a Connection object (no actual network connect).
# py-amqp keeps `host` as the raw "host:port" string until connect() resolves it,
# and folds userid/password into the SASL login_response rather than storing them
# as attributes — so assert on what the pre-connect object actually carries.
from amqp import Connection

conn = Connection(
    host='localhost:5672',
    userid='guest',
    password='guest',
    virtual_host='/',
    connect_timeout=1,
)

assert conn.host == 'localhost:5672'
assert conn.virtual_host == '/'
assert conn.connect_timeout == 1
assert conn.ssl is False
assert not conn.connected
print(f"Connection: host={conn.host} vhost={conn.virtual_host}")
print(f"connected={conn.connected}")

In [ ]:
# Example 2: Exception hierarchy + error-code mapping
# In py-amqp 5.x AccessRefused (403) is a ChannelError subclass
# (IrrecoverableChannelError), matching its reply-code classification.
from amqp.exceptions import (
    AMQPError,
    ConnectionError as AMQPConnectionError,
    ChannelError,
    NotFound,
    AccessRefused,
    ERROR_MAP,
)

assert issubclass(AMQPConnectionError, AMQPError)
assert issubclass(ChannelError, AMQPError)
assert issubclass(NotFound, ChannelError)
assert issubclass(AccessRefused, ChannelError)

# 404 NOT_FOUND, 403 ACCESS_REFUSED — standard AMQP 0.9.1 codes
assert ERROR_MAP[404] is NotFound
assert ERROR_MAP[403] is AccessRefused
print(f"AMQP error code 404 -> {ERROR_MAP[404].__name__}")
print(f"AMQP error code 403 -> {ERROR_MAP[403].__name__}")

In [ ]:
# Example 3: Logger helper + Connection defaults
from amqp.utils import get_logger
from amqp import Connection

# Connection accepts an amqp:// URL via host kwarg parsing in higher-level libs;
# at the py-amqp layer we test the logger helper + Connection arg defaults.
logger = get_logger('amqp.test')
assert logger is not None
assert logger.name.startswith('amqp')

# Default Connection (no args) should pick reasonable defaults; pre-connect the
# host stays the raw "host:port" string and port/userid are not yet attributes.
conn = Connection()
assert conn.host == 'localhost:5672'
assert conn.virtual_host == '/'
assert conn.ssl is False
assert not conn.connected
print(f"logger.name = {logger.name}")
print(f"defaults: {conn.host} vhost={conn.virtual_host}")

In [ ]:
# Example 4: Connection cannot actually open without a broker — verify clean failure
import socket
from amqp import Connection
from amqp.exceptions import AMQPError

# Point at a closed port; connect should fail cleanly, not crash the interpreter.
conn = Connection(host='127.0.0.1:1', connect_timeout=1)
raised = None
try:
    conn.connect()
except (OSError, socket.error, AMQPError, ConnectionError) as e:
    raised = e

assert raised is not None, 'Expected connection to fail against 127.0.0.1:1'
assert not conn.connected
print(f"Got expected failure: {type(raised).__name__}: {str(raised)[:100]}")
print("amqp smoke test PASSED")